In [136]:
import pandas as pd
import numpy as np

In [137]:
forms=pd.read_csv(r"C:\Users\kmvri\OneDrive\Documents\Desktop\CSE 2023-2027\SEM 6\23CSE452-BUSINESS ANALYTICS\AIRLINE CUSTOMER PREFERENCE\cleaned_airline_survey.csv")
new=pd.read_csv(r"C:\Users\kmvri\OneDrive\Documents\Desktop\CSE 2023-2027\SEM 6\23CSE452-BUSINESS ANALYTICS\AIRLINE CUSTOMER PREFERENCE\Indian_Domestic_Airline(kaggle).csv")
forms2=forms.copy()
print(new.columns)
print(forms2.columns)

Index(['AirLine_Name', 'Rating - 10', 'Title', 'Name', 'Date', 'Review',
       'Recommond'],
      dtype='object')
Index(['Gender', 'Age', 'Occupation', 'Purpose_of_Travel', 'Travel_Frequency',
       'Travel_Class', 'Flight_Preference', 'Booking_Mode',
       'Influencing_Factors', 'Airline_Last_Flown', 'Price_Sensitivity',
       'Loyalty_Program', 'Reward_Preference', 'Schedule_Preference',
       'Inflight_Priority'],
      dtype='object')


In [138]:
new = new[['AirLine_Name', 'Rating - 10', 'Review', 'Recommond']]

In [139]:
new.rename(columns={
    'AirLine_Name': 'Airline_Last_Flown',
    'Recommond': 'Loyalty_Program'}, inplace=True)


In [140]:
survey_columns = [
    'Gender', 'Age', 'Occupation', 'Purpose_of_Travel',
    'Travel_Frequency', 'Travel_Class', 'Flight_Preference',
    'Booking_Mode', 'Influencing_Factors', 'Airline_Last_Flown',
    'Price_Sensitivity', 'Loyalty_Program', 'Reward_Preference',
    'Schedule_Preference', 'Inflight_Priority'
]

for col in survey_columns:
    if col not in new.columns:
        new[col] = np.nan


In [141]:
def format_airline_last_flown(cell):
    if pd.isna(cell) or str(cell).strip() == '':
        return 'Other;'
    raw_airlines = [a.strip() for a in str(cell).split(';') if a.strip()]
    mapped = []
    for a in raw_airlines:
        a = a.lower()
        if 'indigo' in a:
            mapped.append('Indigo')
        elif 'air india' in a:
            mapped.append('Air India')
        elif 'spice' in a:
            mapped.append('Spicejet')
        elif 'akasa' in a:
            mapped.append('Akasa')
        elif 'vistara' in a:
            mapped.append('Vistara')
        else:
            mapped.append('Other')
    mapped = list(dict.fromkeys(mapped))
    return ';'.join(mapped) + ';'


In [142]:
new['Airline_Last_Flown'] = new['Airline_Last_Flown'].apply(format_airline_last_flown)


In [143]:
new['Rating - 10'] = pd.to_numeric(new['Rating - 10'], errors='coerce')

new['Price_Sensitivity'] = new['Rating - 10'].apply(
    lambda x: 'Very sensitive' if x < 6 else
              'Somewhat sensitive' if x < 8 else
              'Not sensitive'
)


In [144]:
new['Loyalty_Program'] = new['Loyalty_Program'].str.lower().map({
    'yes': 'Yes',
    'no': 'No'
})


In [145]:
purpose_values = [
    'Leisure/Vacation',
    'Business',
    'Family Visit',
    'Education',
    'Others'
]

In [146]:
def infer_purpose(review):
    text = str(review).lower()
    if 'vacation' in text or 'holiday' in text or 'tour' in text or 'trip' in text:
        return 'Leisure/Vacation'
    if 'business' in text or 'meeting' in text or 'office' in text or 'work' in text:
        return 'Business'
    if 'family' in text or 'parents' in text or 'wedding' in text or 'relatives' in text:
        return 'Family Visit'
    if 'study' in text or 'college' in text or 'university' in text or 'exam' in text:
        return 'Education'
    return np.random.choice(purpose_values)

new['Purpose_of_Travel'] = new['Review'].apply(infer_purpose)


In [147]:
new['Purpose_of_Travel'] = new['Review'].apply(travel_purpose)

In [150]:
gender_values = ['Woman', 'Man']

age_values = [
    '12-18',
    '19-24',
    '25-34',
    '35-44',
    '45-60',
    '60+'
]

In [151]:
import numpy as np
def fill_gender(review):
    text = str(review).lower()
    if any(word in text for word in ['she', 'her', 'wife', 'mother']):
        return 'Woman'
    if any(word in text for word in ['he', 'him', 'husband', 'father']):
        return 'Man'
    return np.random.choice(gender_values)
def fill_age(review):
    text = str(review).lower()
    if any(word in text for word in ['school', 'college', 'university', 'exam']):
        return '19-24'
    if any(word in text for word in ['office', 'work', 'business', 'job']):
        return np.random.choice(['25-34', '35-44'])
    if any(word in text for word in ['retired', 'senior', 'elderly']):
        return '60+'
    return np.random.choice(age_values)

new['Gender'] = new['Review'].apply(fill_gender)
new['Age'] =new['Review'].apply(fill_age)


In [152]:
occupation_values = ['Student','Working Professional','Business Owner','Government Employee','Homemaker','Others']

In [153]:
def fill_occupation(review, age):
    text = str(review).lower()
    if any(word in text for word in ['student', 'college', 'university', 'exam']):
        return 'Student'
    if any(word in text for word in ['office', 'job', 'company', 'working', 'employee']):
        return 'Working Professional'
    if any(word in text for word in ['business', 'startup', 'owner', 'entrepreneur']):
        return 'Business Owner'
    if any(word in text for word in ['government', 'govt', 'public sector', 'psu', 'officer']):
        return 'Government Employee'
    if any(word in text for word in ['housewife', 'homemaker', 'managing home']):
        return 'Homemaker'
    if age in ['12-18', '19-24']:
        return 'Student'
    if age in ['25-34', '35-44']:
        return np.random.choice(['Working Professional', 'Business Owner'])
    if age == '45-60':
        return np.random.choice(['Working Professional', 'Government Employee'])
    if age == '60+':
        return 'Others'

    return np.random.choice(occupation_values)

new['Occupation'] = new.apply(
    lambda row: fill_occupation(row['Review'], row['Age']),
    axis=1
)


In [154]:
travel_class_values = [
    'Economy',
    'Premium Economy',
    'Business',
    'First Class']

In [155]:
def infer_travel_class(review):
    text = str(review).lower()
    if any(word in text for word in ['first class', 'suite', 'luxury cabin']):
        return 'First Class'
    if any(word in text for word in ['business class', 'lie flat', 'lie-flat']):
        return 'Business'
    if any(word in text for word in ['premium economy', 'extra legroom']):
        return 'Premium Economy'
    if any(word in text for word in ['economy', 'coach', 'basic seat']):
        return 'Economy'
    return np.random.choice(travel_class_values)

new['Travel_Class'] = new['Review'].apply(infer_travel_class)


In [156]:
frequency_values = [
    '0 flight',
    '1 flight',
    '2-3 flights',
    '4-6 flights',
    '7-10 flights',
    'More than 10 flight'
]


In [157]:
def fill_travel_frequency(review, travel_class, age):
    text = str(review).lower()
    if any(word in text for word in ['frequent', 'often', 'regularly', 'many times']):
        return np.random.choice(['7-10 flights', 'More than 10 flight'])
    if any(word in text for word in ['first time', 'rarely', 'once']):
        return '1 flight'
    if any(word in text for word in ['twice', 'couple of times']):
        return '2-3 flights'
    if travel_class in ['Business', 'First Class']:
        return np.random.choice(['4-6 flights', '7-10 flights'])
    if age in ['12-18', '60+']:
        return np.random.choice(['0 flight', '1 flight'])
    if age in ['25-34', '35-44']:
        return np.random.choice(['2-3 flights', '4-6 flights'])
    return np.random.choice(frequency_values)

new['Travel_Frequency'] = new.apply(lambda row: fill_travel_frequency(row['Review'],row['Travel_Class'],row['Age']),axis=1)


In [158]:
flight_pref_values = [
    'Direct Flight only',
    'Prefer 1-stop if cheaper',
    'Prefer layover for rest/personal reasons',
    'No preference'
]


In [159]:
def fill_flight_preference(review):
    text = str(review).lower()
    if any(word in text for word in ['missed connection', 'layover hassle', 'long delay', 'connecting flight issue']):
        return 'Direct Flight only'
    if any(word in text for word in ['cheap', 'cheaper', 'low fare', 'save money']):
        return 'Prefer 1-stop if cheaper'
    if any(word in text for word in ['layover', 'long layover', 'rest', 'break', 'lounge']):
        return 'Prefer layover for rest/personal reasons'
    if any(word in text for word in ['no preference', 'doesn’t matter', 'does not matter', 'fine with']):
        return 'No preference'
    return np.random.choice(flight_pref_values)

new['Flight_Preference'] = new['Review'].apply(fill_flight_preference)


In [160]:
booking_mode_values = [
    'Airline Website/App',
    'Third party website (e.g MakeMyTrip, Goibibo)',
    'Travel Agent',
    'Corporate booking portal',
    'Other'
]


In [161]:
def infer_booking_mode(review):
    text = str(review).lower()
    if any(word in text for word in ['airline app', 'official website', 'airline website']):
        return 'Airline Website/App'
    if any(word in text for word in ['makemytrip', 'goibibo', 'cleartrip', 'ixigo']):
        return 'Third party website (e.g MakeMyTrip, Goibibo)'
    if any(word in text for word in ['travel agent', 'agent', 'agency']):
        return 'Travel Agent'
    if any(word in text for word in ['corporate', 'company booking', 'office portal', 'business portal']):
        return 'Corporate booking portal'
    return np.random.choice(booking_mode_values)

new['Booking_Mode'] = new['Review'].apply(infer_booking_mode)


In [162]:
schedule_values = [
    'Early morning',
    'Afternoon',
    'Evening',
    'Late Night',
    'No preference'
]


In [163]:
def fill_schedule_preference(review):
    text = str(review).lower()
    if any(w in text for w in ['early morning', 'morning flight', 'dawn']):
        return 'Early morning'
    if any(w in text for w in ['afternoon', 'noon']):
        return 'Afternoon'
    if any(w in text for w in ['evening', 'night departure']):
        return 'Evening'
    if any(w in text for w in ['late night', 'red eye', 'overnight']):
        return 'Late Night'
    if any(w in text for w in ['no preference', 'anytime', 'flexible']):
        return 'No preference'
    return np.random.choice(schedule_values)
new['Schedule_Preference'] = new['Review'].apply(fill_schedule_preference)


In [164]:
inflight_values = [
    'Extra legroom',
    'Extra baggage facility',
    'Seat comfort',
    'In-flight entertainment',
    'Cabin crew behavior',
    'Amenities(e.g charging ports, eye-mask, blankets)'
]


In [165]:
def fill_inflight_priority(review):
    text = str(review).lower()
    if any(w in text for w in ['legroom', 'space', 'cramped']):
        return 'Extra legroom'
    if any(w in text for w in ['baggage', 'luggage', 'extra bag']):
        return 'Extra baggage facility'
    if any(w in text for w in ['seat', 'comfortable', 'uncomfortable']):
        return 'Seat comfort'
    if any(w in text for w in ['entertainment', 'screen', 'movie']):
        return 'In-flight entertainment'
    if any(w in text for w in ['crew', 'staff', 'attendant']):
        return 'Cabin crew behavior'
    if any(w in text for w in ['charging', 'blanket', 'pillow', 'amenities']):
        return 'Amenities(e.g charging ports, eye-mask, blankets)'
    return np.random.choice(inflight_values)
new['Inflight_Priority'] = new['Review'].apply(fill_inflight_priority)


In [166]:
influencing_values = [
    'Ticket Price',
    'Punctuality',
    'In flight service quality',
    'Loyalty programs',
    'Brand reputation',
    'Safety rating',
    'Free Food'
]
reward_values = [
    'Cashback or Discount',
    'Free seat upgrades',
    'Free lounge Access',
    'Priority Check-in/boarding',
    'Free Flights'
]

In [167]:
import random
def random_multiselect(options, min_k=1, max_k=3):
    k = random.randint(min_k, max_k)
    return random.sample(options, k)
new['Influencing_Factors'] = new.apply(
    lambda _: random_multiselect(influencing_values),
    axis=1
)


In [168]:
new['Reward_Preference'] = new.apply(
    lambda _: random_multiselect(reward_values),
    axis=1
)


In [169]:
final_column_order = ['Gender','Age','Occupation','Purpose_of_Travel','Travel_Frequency','Travel_Class','Flight_Preference','Booking_Mode','Influencing_Factors','Airline_Last_Flown','Price_Sensitivity','Loyalty_Program','Reward_Preference','Schedule_Preference','Inflight_Priority']
new = new[final_column_order]


In [171]:
new.sample(40)

,Gender,Age,Occupation,Purpose_of_Travel,Travel_Frequency,Travel_Class,Flight_Preference,Booking_Mode,Influencing_Factors,Airline_Last_Flown,Price_Sensitivity,Loyalty_Program,Reward_Preference,Schedule_Preference,Inflight_Priority
2099,Woman,35-44,Business Owner,Business,1 flight,Premium Economy,No preference,Travel Agent,"[Brand reputation, Punctuality]",Vistara;,Not sensitive,Yes,[Free Flights],Late Night,Extra legroom
850,Man,25-34,Working Professional,Others,2-3 flights,Premium Economy,Direct Flight only,Other,"[In flight service quality, Loyalty programs, ...",Other;,Very sensitive,No,"[Priority Check-in/boarding, Free seat upgrade...",Afternoon,Extra baggage facility
1663,Man,35-44,Business Owner,Business,1 flight,Business,Prefer 1-stop if cheaper,Travel Agent,"[Safety rating, Ticket Price, Punctuality]",Spicejet;,Very sensitive,No,"[Cashback or Discount, Free lounge Access, Pri...",Early morning,In-flight entertainment
2160,Woman,45-60,Government Employee,Others,4-6 flights,Premium Economy,Prefer layover for rest/personal reasons,Corporate booking portal,"[Ticket Price, Loyalty programs]",Vistara;,Not sensitive,Yes,"[Free seat upgrades, Free lounge Access]",Early morning,Seat comfort
770,Woman,45-60,Working Professional,Others,0 flight,Premium Economy,No preference,Airline Website/App,"[Loyalty programs, Free Food]",Air India;,Very sensitive,No,[Free lounge Access],Early morning,Seat comfort
692,Man,35-44,Working Professional,Others,2-3 flights,Premium Economy,Prefer layover for rest/personal reasons,Travel Agent,"[Brand reputation, Ticket Price, Free Food]",Air India;,Very sensitive,No,[Cashback or Discount],No preference,Cabin crew behavior
768,Woman,12-18,Student,Others,7-10 flights,First Class,Prefer 1-stop if cheaper,"Third party website (e.g MakeMyTrip, Goibibo)","[Ticket Price, Free Food, In flight service qu...",Air India;,Not sensitive,Yes,[Free lounge Access],Afternoon,Extra legroom
894,Man,25-34,Business Owner,Others,4-6 flights,Economy,Prefer 1-stop if cheaper,Other,[Loyalty programs],Other;,Very sensitive,No,"[Cashback or Discount, Priority Check-in/board...",Evening,Extra baggage facility
1248,Woman,45-60,Working Professional,Family Visit,7-10 flights,Economy,Prefer 1-stop if cheaper,"Third party website (e.g MakeMyTrip, Goibibo)","[Punctuality, Brand reputation]",Indigo;,Very sensitive,No,"[Free seat upgrades, Priority Check-in/boarding]",No preference,In-flight entertainment
1920,Woman,25-34,Working Professional,Business,4-6 flights,First Class,Direct Flight only,"Third party website (e.g MakeMyTrip, Goibibo)",[Free Food],Spicejet;,Very sensitive,No,"[Priority Check-in/boarding, Free Flights, Fre...",No preference,Extra legroom


In [172]:
forms2.sample(10)

,Gender,Age,Occupation,Purpose_of_Travel,Travel_Frequency,Travel_Class,Flight_Preference,Booking_Mode,Influencing_Factors,Airline_Last_Flown,Price_Sensitivity,Loyalty_Program,Reward_Preference,Schedule_Preference,Inflight_Priority
8,Man,12-18,Student,Family Visit;Education;,4-6 flights,Economy,Direct Flight only,"Third party website (e.g MakeMyTrip, Goibibo)","['Ticket Price', 'Free Food', 'Punctuality']",Air India;,Very sensitive,No,"['Free seat upgrades', 'Free lounge Access', '...",Afternoon,"['Extra legroom', 'Extra baggage facility', 'S..."
136,Woman,19-24,Student,Leisure/Vacation,0 flight,Economy,Direct Flight only,Airline Website/App,"['Ticket Price', 'Safety rating', 'Free Food']",Other,Very sensitive,No,"['Extra baggage', 'Cashback or Discount', 'Pri...",No preference,"['Extra baggage facility', 'Seat comfort', 'Am..."
98,Man,45-60,Working Professional,Business;Family Visit;Leisure/Vacation;,7-10 flights,Economy,Direct Flight only,"Third party website (e.g MakeMyTrip, Goibibo)","['Ticket Price', 'Brand reputation', 'In fligh...",Indigo;Air India;Other;,Somewhat sensitive,Yes,"['Free lounge Access', 'Free Flights', 'Cashba...",Evening,"['In-flight entertainment', 'Seat comfort', 'E..."
132,Man,19-24,Working Professional,Business;Family Visit;Leisure/Vacation,4-6 flights,Economy,Direct Flight only,Airline Website/App,"['Ticket Price', 'In flight service quality', ...",Indigo;Air India;Akasa,Somewhat sensitive,Yes,"['Free seat upgrades', 'Priority Check-in/boar...",No preference,"['Extra legroom', 'Seat comfort', 'In-flight e..."
43,Man,19-24,Student,Leisure/Vacation;Family Visit;Education;Others...,7-10 flights,Premium Economy,Prefer 1-stop if cheaper,Airline Website/App,"['Ticket Price', 'Seat comfort', 'Safety rating']",Indigo;Air India;Vistara;Akasa;Other;,Somewhat sensitive,Yes,"['Free seat upgrades', 'Extra baggage', 'Prior...",Late Night,"['Extra legroom', 'Extra baggage facility', 'S..."
356,Woman,35-44,Homemaker,Leisure/Vacation;Family Visit;Others,2-3 flights,Economy,Prefer 1-stop if cheaper,"Third party website (e.g MakeMyTrip, Goibibo)","['Ticket Price', 'Safety rating', 'In flight s...",Indigo;Spicejet;Air India,Very sensitive,No,"['Free seat upgrades', 'Cashback or Discount',...",Afternoon,"['Cabin crew behavior', 'Seat comfort', 'Extra..."
199,Woman,19-24,Student,Education;Leisure/Vacation;Family Visit,7-10 flights,Economy,Direct Flight only,"Third party website (e.g MakeMyTrip, Goibibo)","['Ticket Price', 'Punctuality', 'Safety rating']",Air India;Indigo;Vistara,Somewhat sensitive,No,"['Cashback or Discount', 'Free lounge Access',...",Evening,"['Cabin crew behavior', 'Seat comfort', 'Extra..."
118,Man,19-24,Student,Leisure/Vacation;Family Visit;Education,0 flight,Economy,Direct Flight only,Airline Website/App,"['In flight service quality', 'Ticket Price', ...",Other,Somewhat sensitive,No,"['Free Flights', 'Free seat upgrades', 'Cashba...",Early morning,"['Amenities(e.g charging ports, eye-mask, blan..."
323,Man,35-44,Working Professional,Family Visit;Education,4-6 flights,Economy,Prefer 1-stop if cheaper,"Third party website (e.g MakeMyTrip, Goibibo)","['Ticket Price', 'Punctuality', 'Safety rating']",Indigo,Somewhat sensitive,No,"['Free Flights', 'Cashback or Discount', 'Free...",No preference,"['Amenities(e.g charging ports, eye-mask, blan..."
14,Woman,25-34,Student,Leisure/Vacation;,1 flight,Economy,Direct Flight only,Airline Website/App,"['Ticket Price', 'Punctuality', 'In flight ser...",Spicejet;,Not sensitive,No,"['Cashback or Discount', 'Extra baggage', 'Fre...",Evening,"['Amenities(e.g charging ports, eye-mask, blan..."


In [174]:
assert forms2.columns.tolist() == new.columns.tolist(), \
    "Column mismatch! Fix before copying."


In [175]:
final_df = pd.concat([forms2, new],axis=0,ignore_index=True)


In [176]:
print("Survey rows:", len(forms2))
print("Kaggle rows:", len(new))
print("Final rows:", len(final_df))


Survey rows: 358
Kaggle rows: 2210
Final rows: 2568


In [177]:
final_df.to_csv("combined_dataset.csv", index=False)
